In [72]:
##Arxiv--Reserch
##Tool creati
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [73]:
#use the inbulit tool of wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki =WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [74]:
api_wrapper_arxiv= ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [75]:
tools = [wiki,arxiv]

In [76]:
##Creat Custome tools[Rag tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


In [77]:
import os
os.environ["HUGGINGFACE"]= os.getenv("HUGGINGFACE")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6075.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [78]:

loader = WebBaseLoader("https://docs.langchain.com/")
docs = loader.load()
document = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordp=FAISS.from_documents(document,embeddings)
retriever = vectordp.as_retriever()
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B274391510>)

In [79]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"Langsmith-search","serch any information about langsmith")
retriever_tool.name

'Langsmith-search'

In [80]:
tools = [wiki,arxiv,retriever_tool]

In [ ]:
##Run all the tools with agents and LLM models
##Tool,LLm-->AgentExecutor
from langchain_groq import ChatGroq
groq = os.getenv('GROQ')

llm = ChatGroq(api_key=groq,model="llama-3.1-8b-instant")





: 

In [82]:
##prompt Template
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages



[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant')),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [87]:
#agents
from langchain.agents import create_openai_tools_agent
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'chat_history': []}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-functions-agent', 'lc_hub_commit_has

AgentExecutor(verbose=True, tags=['zero-shot-react-description'], agent=ZeroShotAgent(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['agent_scratchpad', 'input', 'page_content'], template="Answer the following questions as best you can. You have access to the following tools:\n\nwikipedia - A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\narxiv - A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.\nLangsmith-search(query: 'str', *, retriever: 'BaseRetriever' = VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B274391510>)

In [88]:
#agent excuter
from langchain.agents import AgentExecutor
agent_excute=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_excute

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'chat_history': []}, metadata={'lc_hub_owner': 'hwc

<bound method Chain.run of AgentExecutor(verbose=True, agent=RunnableAgent(runnable=AgentExecutor(verbose=True, tags=['zero-shot-react-description'], agent=ZeroShotAgent(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['agent_scratchpad', 'input', 'page_content'], template="Answer the following questions as best you can. You have access to the following tools:\n\nwikipedia - A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\narxiv - A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.\nLangsmith-search(query: 'str', *, retriever: 'BaseRetriever' = VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vect

In [89]:
agent_excute.invoke({
    "input": "What is LangChain?"
})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `Langsmith-search` with `{'query': 'LangChain'}`


Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration features.Learn moreDeploymentShip your agent in one click, using scalable infrastructure built for long-running tasks.Learn moreLangSmit

{'input': 'What is LangChain?',
 'output': 'It appears that the summary was cut off. Let me try again.\n\n <function=wikipedia>{"query": "LangChain"}</function>'}

In [91]:
agent_excute.invoke({
    "input": "What is the meaning of saad?"
})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `wikipedia` with `{'query': 'saad meaning'}`


Page: Saad (name)
Summary: Saad (Arabic: سعد, romanized: Saʿd) is a common male Arabic given name. The name stems from the Arabic verb sa‘ada (‏سَعَدَ‎ 'to be happy, fortunate or lucky'). 
Saad is the stem of variant given names Suad and Sa‘id.
It ma
Invoking: `wikipedia` with `{'query': 'saad meaning'}`
responded: It seems like the response was cut off. I'll try to provide a complete response.



Page: Saad (name)
Summary: Saad (Arabic: سعد, romanized: Saʿd) is a common male Arabic given name. The name stems from the Arabic verb sa‘ada (‏سَعَدَ‎ 'to be happy, fortunate or lucky'). 
Saad is the stem of variant given names Suad and Sa‘id.
It ma
Invoking: `wikipedia` with `{'query': 'Saad meaning'}`


Page: Saad (name)
Summary: Saad (Arabic: سعد, romanized: Saʿd) is a common male Arabic given name. The name stems from the Arabic verb sa‘ada (‏سَعَدَ‎ 'to be happy, fortunate or lucky'). 
Saad is the stem of variant given names Suad

{'input': 'What is the meaning of saad?',
 'output': 'The final answer to the user\'s question "What is the meaning of saad?" is:\n\nSaad (Arabic: سعد, romanized: Saʿd) is a common male Arabic given name. The name stems from the Arabic verb sa‘ada (\u200fسَعَدَ\u200e \'to be happy, fortunate or lucky\'). \nSaad is the stem of variant given names Suad and Sa‘id.\nIt means \'lucky\' or \'fortunate\'.'}